# Inference & Evaluation

Load trained models and evaluate with detailed metrics and visualizations.

In [ ]:
import sys; sys.path.insert(0, '..')
from src import suppress_logs; suppress_logs()

import torch
torch.set_float32_matmul_precision('medium')

import numpy as np
from sklearn.metrics import accuracy_score, f1_score, classification_report
from pytorch_lightning import seed_everything
from src.data import GestureDataModule
from src.hub import get_model
from src.models import BiLSTMModule, TransformerModule
from src.visualization import plot_confusion_matrices, print_results_table

## Configuration

In [ ]:
DATA_PATH = None  # Auto-download from HuggingFace
TEST_SPLIT = 0.2
SEED = 42

seed_everything(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## Load Data & Models

In [ ]:
dm = GestureDataModule(data_path=DATA_PATH, test_split=TEST_SPLIT, seed=SEED)
dm.setup()
print(f'Classes: {dm.class_names}')
print(f'Test samples: {len(dm.test_dataset)}')

In [ ]:
models = {}

# Load models (local checkpoint or HuggingFace Hub)
try:
    models['bilstm'] = get_model('bilstm').to(device).eval()
    print('✓ BiLSTM loaded')
except Exception as e:
    print(f'✗ BiLSTM: {e}')

try:
    models['transformer'] = get_model('transformer').to(device).eval()
    print('✓ Transformer loaded')
except Exception as e:
    print(f'✗ Transformer: {e}')

## Run Inference

In [ ]:
predictions = {}
results = {}

for name, model in models.items():
    preds, targets = [], []
    with torch.no_grad():
        for x, y in dm.test_dataloader():
            out = model(x.to(device))
            preds.extend(out.argmax(1).cpu().numpy())
            targets.extend(y.numpy())
    
    predictions[name] = (preds, targets)
    acc = accuracy_score(targets, preds)
    f1 = f1_score(targets, preds, average='macro')
    results[name] = {'accuracy': acc, 'f1': f1}
    print(f'{name.upper()}: Acc={acc:.4f}, F1={f1:.4f}')

## Results

In [ ]:
print_results_table(results)

In [ ]:
fig = plot_confusion_matrices(predictions, dm.class_names)
Path('../plots').mkdir(exist_ok=True)
fig.savefig('../plots/confusion_matrices.png', dpi=150)

## Classification Reports

In [ ]:
for name, (preds, targets) in predictions.items():
    print(f'\n{"="*60}\n{name.upper()} Classification Report\n{"="*60}')
    print(classification_report(targets, preds, target_names=dm.class_names))

## Per-Class Comparison

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(dm.class_names))
width = 0.35

for i, (name, (preds, targets)) in enumerate(predictions.items()):
    cm = confusion_matrix(targets, preds)
    per_class_acc = cm.diagonal() / cm.sum(axis=1)
    ax.bar(x + i * width, per_class_acc, width, label=name.upper())

ax.set_xlabel('Class')
ax.set_ylabel('Accuracy')
ax.set_title('Per-Class Accuracy Comparison')
ax.set_xticks(x + width / 2)
ax.set_xticklabels(dm.class_names, rotation=45, ha='right')
ax.legend()
ax.set_ylim(0, 1.1)
plt.tight_layout()
plt.savefig('../plots/per_class_comparison.png', dpi=150)